Checking the Streaming data folder for all JSON

In [0]:
dbutils.fs.ls('databricks-datasets/structured-streaming/events')

Out[19]: [FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-0.json', name='file-0.json', size=72530, modificationTime=1469673865000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-1.json', name='file-1.json', size=72961, modificationTime=1469673866000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-10.json', name='file-10.json', size=73025, modificationTime=1469673878000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-11.json', name='file-11.json', size=72999, modificationTime=1469673879000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-12.json', name='file-12.json', size=72987, modificationTime=1469673880000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-13.json', name='file-13.json', size=73006, modificationTime=1469673881000),
 FileInfo(path='dbfs:/databricks-datasets/structured-streaming/events/file-14.json', name

Setting the input folder for structred streaming input directory---

In [0]:
inputPath = "/databricks-datasets/structured-streaming/events/"

Defining the type for input 

In [0]:
from pyspark.sql.types import StructType,StructField,TimestampType,StringType
jsonSchema = StructType([ StructField("time", TimestampType(), True), StructField("action", StringType(), True) ])


Creating a Dataframe which will take input from input path and take one file at a time 

In [0]:
streamingInputDF = (
  spark
    .readStream
    .schema(jsonSchema)               # Set the schema of the JSON data
    .option("maxFilesPerTrigger", 1)  # Treat a sequence of files as a stream by picking one file at a time
    .json(inputPath)
)


Viewing Dataframe

In [0]:
display(streamingInputDF)

Creating a dataframe which will take the count from main dataframe and it will refresh every five minutes

In [0]:
from pyspark.sql.functions import window,count
from pyspark.sql import SparkSession
streamingCountsDF = (
  streamingInputDF
    .groupBy("action",window("time", "5 minutes"))
    .count()
)

In [0]:
display(streamingCountsDF)

action,window,count
Open,"List(2016-07-26T21:15:00.000+0000, 2016-07-26T21:20:00.000+0000)",74
Close,"List(2016-07-27T07:50:00.000+0000, 2016-07-27T07:55:00.000+0000)",75
Open,"List(2016-07-26T08:20:00.000+0000, 2016-07-26T08:25:00.000+0000)",95
Open,"List(2016-07-27T04:45:00.000+0000, 2016-07-27T04:50:00.000+0000)",87
Close,"List(2016-07-27T09:50:00.000+0000, 2016-07-27T09:55:00.000+0000)",90
Open,"List(2016-07-27T10:00:00.000+0000, 2016-07-27T10:05:00.000+0000)",83
Close,"List(2016-07-26T16:50:00.000+0000, 2016-07-26T16:55:00.000+0000)",90
Close,"List(2016-07-26T17:45:00.000+0000, 2016-07-26T17:50:00.000+0000)",80
Open,"List(2016-07-27T13:55:00.000+0000, 2016-07-27T14:00:00.000+0000)",88
Open,"List(2016-07-27T01:30:00.000+0000, 2016-07-27T01:35:00.000+0000)",79


In [0]:
type(streamingInputDF)

Out[20]: pyspark.sql.dataframe.DataFrame

it will exposed in  memory and we can run our sql statement

In [0]:
query = (
  streamingCountsDF
    .writeStream
    .format("memory")        # memory = store in-memory table (for testing only)
    .queryName("counts")     # counts = name of the in-memory table
    .outputMode("complete")  # complete = all the counts should be in the table
    .start()
)

The tablename exposed as counts

In [0]:
%sql select action, date_format(window.end, "MMM-dd HH:mm") as time, count from counts order by time, action

action,time,count
Open,Jul-26 02:50,32
Close,Jul-26 02:55,5
Open,Jul-26 02:55,66
Close,Jul-26 03:00,6
Open,Jul-26 03:00,81
Close,Jul-26 03:05,5
Open,Jul-26 03:05,86
Close,Jul-26 03:10,14
Open,Jul-26 03:10,76
Close,Jul-26 03:15,16


In [0]:
%sql
select count(1) from counts

count(1)
1226
